<a href="https://colab.research.google.com/github/mohamadfaisalbashir/scikit-learn-cookbook/blob/main/02_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Common Conventions and API Elements of scikit-learn**
This notebook covers The impact of raw data on model performance:

1. The impact of raw data on model performance
2. Handling missing data
3. Scaling techniques
4. Encoding categorical variables
5. Introduction to pipelines in scikit-learn
6. Feature engineering
7. Practical exercises on data preprocessing

## **1. The impact of raw data on model performance**

Machine learning algorithms learn patterns from data, meaning the quality of your input data is the most critical element in determining the model's success. If the input data is flawed, the model's ability to generalize on unseen data diminishes, which perfectly illustrates the concept of "garbage in, garbage out".

Common data quality issues that negatively impact models include:
- **Missing data:** Incomplete datasets can cause the model to make erroneous assumptions or fail to execute entirely.

- **Outliers:** Extreme values can distort statistical analyses.

- **Categorical variables:** Most ML algorithms require numerical input, so categories must be transformed.

- **Feature scaling:** Features with different scales can hinder model convergence, especially for distance-based algorithms.

- **Data leakage:** Using outside information from the testing dataset during model training leads to overly optimistic and invalid performance metrics.

## **Creating a Dataset with Missing Values**

In [1]:
import numpy as np
import pandas as pd

np.random.seed(2024)  # For reproducibility
n_samples = 20
n_features = 10

# Generate random data in [0, 100]
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}
df = pd.DataFrame(data)

# Randomly introduce ~20% missing values
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

# Report missingness
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
print(f"Dataset shape: {df.shape[0]} samples x {df.shape[1]} features")
print(f"Total cells: {total_cells}")
print(f"Missing cells: {int(missing_cells)} ({missing_cells/total_cells*100:.1f}%)")
print(f"Missing per feature:")
print(df.isna().sum().to_string())
print()
print("First 6 rows:")
print(df.head(6).to_string())

Dataset shape: 20 samples x 10 features
Total cells: 200
Missing cells: 44 (22.0%)
Missing per feature:
Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3

First 6 rows:
    Feature1   Feature2   Feature3   Feature4   Feature5   Feature6   Feature7   Feature8   Feature9  Feature10
0  58.801452        NaN  42.009814  34.680397  49.962259        NaN        NaN  89.954588  41.152421        NaN
1  69.910875   9.554215   6.436369  31.287816  37.966499        NaN  82.005649        NaN  75.797616  91.382492
2        NaN  96.090974  59.643269  84.710402        NaN  84.109027        NaN  70.288128   1.778343   4.117690
3        NaN  25.176729  83.732372  88.023110  16.886931  97.205554  84.696823        NaN        NaN  80.077973
4  20.501895        NaN  89.248639  67.655865  58.635861  78.225721  60.911562        NaN  65.114243  99.119187
5  10.606287  76.825393  20.052744   5.367515        NaN  1

## **2. Handling missing data**

Most machine learning algorithms cannot handle missing values directly. While dropping records is sometimes an option, scikit-learn provides several strategies to impute (fill in) missing gaps using estimated values based on the available data.

**SimpleImputer():** Replaces missing entries with a straightforward statistic like the mean, median, or most frequent value.

In [2]:
from sklearn.impute import SimpleImputer

# Initialize with mean strategy
imputer = SimpleImputer(strategy="mean")

# Fit (learn column means) and transform (fill NaNs)
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

print("Learned means for each feature:")
for feat, mean_val in zip(df.columns, imputer.statistics_):
    print(f"  {feat}: {mean_val:.4f}")
print()
print("Missing values after imputation:", int(imputed_df.isna().sum().sum()))
print()
print("First 6 rows after SimpleImputer (mean):")
print(imputed_df.head(6).to_string())

Learned means for each feature:
  Feature1: 52.5586
  Feature2: 53.8647
  Feature3: 52.8975
  Feature4: 54.9616
  Feature5: 46.0559
  Feature6: 57.8230
  Feature7: 51.1450
  Feature8: 50.7150
  Feature9: 60.6167
  Feature10: 48.4000

Missing values after imputation: 0

First 6 rows after SimpleImputer (mean):
    Feature1   Feature2   Feature3   Feature4   Feature5   Feature6   Feature7   Feature8   Feature9  Feature10
0  58.801452  53.864661  42.009814  34.680397  49.962259  57.822964  51.145011  89.954588  41.152421  48.400044
1  69.910875   9.554215   6.436369  31.287816  37.966499  57.822964  82.005649  50.715003  75.797616  91.382492
2  52.558605  96.090974  59.643269  84.710402  46.055883  84.109027  51.145011  70.288128   1.778343   4.117690
3  52.558605  25.176729  83.732372  88.023110  16.886931  97.205554  84.696823  50.715003  60.616723  80.077973
4  20.501895  53.864661  89.248639  67.655865  58.635861  78.225721  60.911562  50.715003  65.114243  99.119187
5  10.606287  76.

**KNNImputer():** Uses the k-nearest neighbors algorithm to fill in values based on neighboring samples, capturing more nuanced relationships.

In [3]:
from sklearn.impute import KNNImputer

# Use 2 nearest neighbors
knn_imputer = KNNImputer(n_neighbors=2)

# Fit and transform
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)

print("Missing values after KNN imputation:", int(knn_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after KNNImputer (k=2):")
print(knn_imputed_df.head(6).to_string())
print()
# Compare imputed values for Feature1 where original was NaN
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Feature1 was NaN at indices: {f1_missing_idx}")
    print(f"  SimpleImputer filled with: {[f'{imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer filled with:    {[f'{knn_imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")

Missing values after KNN imputation: 0

First 6 rows after KNNImputer (k=2):
    Feature1   Feature2   Feature3   Feature4   Feature5   Feature6   Feature7   Feature8   Feature9  Feature10
0  58.801452  48.954043  42.009814  34.680397  49.962259  93.910386  52.549271  89.954588  41.152421  59.833678
1  69.910875   9.554215   6.436369  31.287816  37.966499  86.793468  82.005649  45.416191  75.797616  91.382492
2  67.752344  96.090974  59.643269  84.710402  73.338309  84.109027  59.012876  70.288128   1.778343   4.117690
3  47.880864  25.176729  83.732372  88.023110  16.886931  97.205554  84.696823  64.510281  39.726833  80.077973
4  20.501895  31.670912  89.248639  67.655865  58.635861  78.225721  60.911562  25.903612  65.114243  99.119187
5  10.606287  76.825393  20.052744   5.367515  59.716773  19.703051  34.423301  93.399494  72.206680  12.640276

Feature1 was NaN at indices: [2, 3, 7]
  SimpleImputer filled with: ['52.5586', '52.5586', '52.5586']
  KNNImputer filled with:    ['67.75

**IterativeImputer():** An advanced approach that models each feature with missing values as a regression function of the other features.

In [4]:
from sklearn.experimental import enable_iterative_imputer  # Required
from sklearn.impute import IterativeImputer

# Initialize IterativeImputer (uses BayesianRidge by default)
iterative_imputer = IterativeImputer(random_state=2024, max_iter=10)

# Fit and transform
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)

print("Missing values after IterativeImputer:", int(iterative_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after IterativeImputer:")
print(iterative_imputed_df.head(6).to_string())
print()
# Compare all three methods on Feature1 missing indices
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Comparison of imputed values for Feature1 (NaN indices: {f1_missing_idx}):")
    print(f"  SimpleImputer (mean):  {[f'{imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer (k=2):      {[f'{knn_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  IterativeImputer:      {[f'{iterative_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")

Missing values after IterativeImputer: 0

First 6 rows after IterativeImputer:
    Feature1   Feature2   Feature3   Feature4   Feature5   Feature6   Feature7   Feature8   Feature9  Feature10
0  58.801452  54.365008  42.009814  34.680397  49.962259  58.680582  51.178102  89.954588  41.152421  48.292275
1  69.910875   9.554215   6.436369  31.287816  37.966499  60.070634  82.005649  50.710323  75.797616  91.382492
2  52.539676  96.090974  59.643269  84.710402  46.121290  84.109027  51.155371  70.288128   1.778343   4.117690
3  52.630591  25.176729  83.732372  88.023110  16.886931  97.205554  84.696823  50.680265  50.650877  80.077973
4  20.501895  53.198954  89.248639  67.655865  58.635861  78.225721  60.911562  50.691674  65.114243  99.119187
5  10.606287  76.825393  20.052744   5.367515  46.156310  19.703051  34.423301  93.399494  72.206680  12.640276

Comparison of imputed values for Feature1 (NaN indices: [2, 3, 7]):
  SimpleImputer (mean):  ['52.56', '52.56', '52.56']
  KNNImputer (k

## **3. Scaling techniques**

Features in a dataset often have vastly different scales (e.g., age ranging from 0 to 100, while income ranges from 0 to 100,000). Scaling ensures that no single feature dominates the learning process.

**StandardScaler() (Standardization):** Transforms features to have a mean of 0 and a standard deviation of 1. It is highly effective when data follows a Gaussian (normal) distribution.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Means (mu):  {np.round(scaler.mean_, 2)}")
print(f"  Stds (sigma): {np.round(scaler.scale_, 2)}")
print()
print("After scaling -- column statistics:")
print(f"  Means:  {np.round(scaled_df.mean().values, 10)}")
print(f"  Stds:   {np.round(scaled_df.std(ddof=0).values, 4)}")
print()
print("First 6 rows after StandardScaler:")
print(scaled_df.head(6).to_string())

Learned parameters:
  Means (mu):  [52.56 53.9  52.9  54.96 46.06 57.86 51.14 50.72 60.03 48.4 ]
  Stds (sigma): [22.95 23.65 29.18 22.68 20.66 25.69 21.63 23.3  24.17 30.08]

After scaling -- column statistics:
  Means:  [-0.  0.  0. -0.  0. -0. -0. -0.  0. -0.]
  Stds:   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after StandardScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0  0.271931  0.019495 -0.373057 -0.894258  0.188761  0.031844  0.001905  1.683800 -0.780898  -0.003574
1  0.756048 -1.874939 -1.592196 -1.043864 -0.391744  0.085949  1.426871 -0.000534  0.652484   1.429056
2 -0.000939  1.783513  0.231259  1.311954  0.002887  1.021588  0.000854  0.839730 -2.409929  -1.472256
3  0.003022 -1.214477  1.056818  1.458037 -1.411839  1.531340  1.551267 -0.001824 -0.387917   1.053212
4 -1.397054 -0.029802  1.245866  0.559887  0.608499  0.792594  0.451822 -0.001334  0.210478   1.686279
5 -1.828276  0.969036 -1.125549 -2.18689

**MinMaxScaler():** Rescales features to a fixed range, typically [0, 1]. It preserves the relationships between values while ensuring equal feature contribution.

In [6]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()  # Default range [0, 1]
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Min per feature: {np.round(minmax_scaler.data_min_, 2)}")
print(f"  Max per feature: {np.round(minmax_scaler.data_max_, 2)}")
print()
print("After scaling -- value range per column:")
print(f"  Min: {np.round(minmax_scaled_df.min().values, 4)}")
print(f"  Max: {np.round(minmax_scaled_df.max().values, 4)}")
print()
print("First 6 rows after MinMaxScaler:")
print(minmax_scaled_df.head(6).to_string())

Learned parameters:
  Min per feature: [ 1.91  9.55  6.4   5.37  6.19 10.46 17.49  0.88  1.78  4.12]
  Max per feature: [96.18 96.09 99.97 88.02 83.11 97.21 94.93 93.4  99.69 99.12]

After scaling -- value range per column:
  Min: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  Max: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after MinMaxScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0  0.603506  0.517824  0.380540  0.354639  0.569020  0.555896  0.435006  0.962767  0.402156   0.464988
1  0.721357  0.000000  0.000369  0.313594  0.413077  0.571920  0.833074  0.538604  0.756013   0.918562
2  0.537080  1.000000  0.568987  0.959922  0.519088  0.849027  0.434713  0.750206  0.000000   0.000000
3  0.538045  0.180530  0.826426  1.000000  0.139045  1.000000  0.867825  0.538279  0.499171   0.799569
4  0.197218  0.504349  0.885377  0.753589  0.681776  0.781206  0.560692  0.538402  0.646896   1.000000
5  0.092244  0.777371  0.145886  0.000000  0.5

**Normalizer():** Scales individual samples to have a unit norm. It is particularly useful for sparse data or when treating each sample magnitude equally.

In [7]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer(norm='l2')  # Default: L2 normalization
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)

# Verify: each ROW should have unit L2 norm
row_norms = np.sqrt((normalized_df ** 2).sum(axis=1))
print("L2 norm of each row (should all be 1.0):")
print(np.round(row_norms.values, 6))
print()
print("First 6 rows after Normalizer (L2):")
print(normalized_df.head(6).to_string())

L2 norm of each row (should all be 1.0):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after Normalizer (L2):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0  0.339168  0.313579  0.242313  0.200037  0.288183  0.338471  0.295196  0.518860  0.237368   0.278551
1  0.376706  0.051482  0.034682  0.168591  0.204578  0.323683  0.441878  0.273247  0.408426   0.492404
2  0.264336  0.483450  0.300075  0.426192  0.232044  0.423166  0.257371  0.353631  0.008947   0.020717
3  0.243762  0.116607  0.387811  0.407684  0.078213  0.450213  0.392278  0.234729  0.234592   0.370886
4  0.095909  0.248868  0.417511  0.316499  0.274302  0.365945  0.284948  0.237139  0.304609   0.463686
5  0.068115  0.493382  0.128781  0.034471  0.296421  0.126535  0.221071  0.599823  0.463720   0.081177


## **4. Encoding categorical variables**

Categorical variables represent discrete values (e.g., colors, brands) and are divided into nominal variables (no intrinsic order) and ordinal variables (clear ordering). Because models require numerical inputs, we must encode these categories.

## **Creating a Categorical Dataset**

In [8]:
import numpy as np
import pandas as pd

np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

print(f"Shape: {categorical_data.shape}")
print(f"Unique values per column:")
for col in categorical_data.columns:
    print(f"  {col}: {sorted(categorical_data[col].unique())} ({categorical_data[col].nunique()} unique)")
print()
print("First 8 rows:")
print(categorical_data.head(8).to_string())

Shape: (20, 3)
Unique values per column:
  Department: ['A', 'B', 'C', 'D'] (4 unique)
  Position: ['Junior', 'Manager', 'Senior'] (3 unique)
  Location: ['CHI', 'LA', 'NY', 'SF'] (4 unique)

First 8 rows:
  Department Position Location
0          A  Manager       LA
1          C  Manager       LA
2          A   Junior       NY
3          A  Manager       LA
4          D  Manager       NY
5          A   Senior       LA
6          C  Manager       SF
7          D  Manager       LA


**OneHotEncoder():** Ideal for nominal data. It converts categories into binary columns (1s and 0s), treating each category independently.

In [9]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)

onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data,
    columns=onehot_encoder.get_feature_names_out()
)

print(f"Original shape: {categorical_data.shape}")
print(f"Encoded shape:  {onehot_encoded_df.shape}")
print(f"New columns: {list(onehot_encoded_df.columns)}")
print()
print("First 8 rows (showing subset of columns):")
print(onehot_encoded_df.head(8).to_string())

Original shape: (20, 3)
Encoded shape:  (20, 11)
New columns: ['Department_A', 'Department_B', 'Department_C', 'Department_D', 'Position_Junior', 'Position_Manager', 'Position_Senior', 'Location_CHI', 'Location_LA', 'Location_NY', 'Location_SF']

First 8 rows (showing subset of columns):
   Department_A  Department_B  Department_C  Department_D  Position_Junior  Position_Manager  Position_Senior  Location_CHI  Location_LA  Location_NY  Location_SF
0           1.0           0.0           0.0           0.0              0.0               1.0              0.0           0.0          1.0          0.0          0.0
1           0.0           0.0           1.0           0.0              0.0               1.0              0.0           0.0          1.0          0.0          0.0
2           1.0           0.0           0.0           0.0              1.0               0.0              0.0           0.0          0.0          1.0          0.0
3           1.0           0.0           0.0           0.0  

**LabelEncoder():** Assigns a unique integer to each category. It is efficient but can mislead algorithms into assuming an ordinal relationship where none exists.

In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()

for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )
    print(f"{column} mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

print()
print("First 8 rows after LabelEncoder:")
print(label_encoded_df.head(8).to_string())

Department mapping: {'A': np.int64(0), 'B': np.int64(1), 'C': np.int64(2), 'D': np.int64(3)}
Position mapping: {'Junior': np.int64(0), 'Manager': np.int64(1), 'Senior': np.int64(2)}
Location mapping: {'CHI': np.int64(0), 'LA': np.int64(1), 'NY': np.int64(2), 'SF': np.int64(3)}

First 8 rows after LabelEncoder:
   Department_encoded  Position_encoded  Location_encoded
0                   0                 1                 1
1                   2                 1                 1
2                   0                 0                 2
3                   0                 1                 1
4                   3                 1                 2
5                   0                 2                 1
6                   2                 1                 3
7                   3                 1                 1


**ColumnTransformer():** Allows the seamless application of different preprocessing techniques across specific columns (e.g., applying one-hot encoding to categories while scaling numerical columns).

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

print("Original mixed dataset:")
print(mixed_data.head(8).to_string())
print(f"\nDtypes:\n{mixed_data.dtypes.to_string()}")

Original mixed dataset:
   Age    Salary  Experience Department Position
0   33  59420.36          12    Finance  Manager
1   57  82895.92          17      Sales   Junior
2   25  38165.76          16    Finance  Manager
3   52  38242.36           7         IT   Junior
4   61  55088.65           8      Sales  Manager
5   26  78688.88           9      Sales   Senior
6   60  49585.62          14    Finance   Junior
7   35  47992.58          17         HR  Manager

Dtypes:
Age             int64
Salary        float64
Experience      int64
Department     object
Position       object


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define which columns get which transformation
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# Fit and transform
transformed_data = column_transformer.fit_transform(mixed_data)

# Build column names for the output
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = (
    column_transformer.named_transformers_["cat"]
    .get_feature_names_out(["Department", "Position"])
)
all_cols = numeric_cols + list(categorical_cols)

transformed_df = pd.DataFrame(transformed_data, columns=all_cols)

print(f"Original shape: {mixed_data.shape}")
print(f"Transformed shape: {transformed_df.shape}")
print(f"\nColumns: {list(transformed_df.columns)}")
print(f"\nFirst 8 rows:")
print(transformed_df.head(8).to_string())

Original shape: (20, 5)
Transformed shape: (20, 10)

Columns: ['Age_scaled', 'Salary_scaled', 'Experience_scaled', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Sales', 'Position_Junior', 'Position_Manager', 'Position_Senior']

First 8 rows:
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager  Position_Senior
0   -1.045349      -0.305043           0.327303                 1.0            0.0            0.0               0.0              0.0               1.0              0.0
1    0.727681       1.105091           1.262454                 0.0            0.0            0.0               1.0              1.0               0.0              0.0
2   -1.636358      -1.581768           1.075424                 1.0            0.0            0.0               0.0              0.0               1.0              0.0
3    0.358300      -1.577167          -0.607848                 

## **5. Introduction to pipelines in scikit-learn**

The Pipeline() class streamlines the complex workflow of data preprocessing and model training into a single object. This sequential execution ensures code simplification, consistency, and prevents data leakage by applying the exact same transformations to training and testing datasets.

Crucially, you must split your data into training and test sets before applying any preprocessing techniques. If preprocessing is applied to the entire dataset prior to splitting, information from the test set leaks into the model, leading to inflated performance metrics.

### **Building a Pipeline**

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = transformed_df.iloc[:, :-1] # all columns except last
y = transformed_df.iloc[:, -1] # last column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

pipeline = Pipeline([
    (
        "imputer", SimpleImputer(strategy="mean")
    ),  # handle missing values
    (
        "scaler", StandardScaler()
    ),  # scale the features
])

X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=X_train.columns,
    index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=X_test.columns,
    index=X_test.index
)

### **Visualizing pipelines**
scikit-learn also allows you to visualize your pipeline using set_config() and display, which
can help in understanding the flow of data through various transformations.

In [14]:
from sklearn import set_config

set_config(display="diagram")
pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

## **6. Feature engineering**
Feature engineering is really an umbrella term that generally refers to two main activities: feature
extraction and feature selection. Effective feature engineering can significantly enhance model
performance by providing algorithms with more informative inputs and reducing or removing noisy and/
or uninformative ones. These recipes will teach common approaches to feature engineering using existing
features to generate new features that may (“may” being the keyword) improve model performance.


**PolynomialFeatures():** Generates polynomial and interaction features from existing numeric ones.

In [15]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=True)

# Use the pipeline-transformed training data
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features,
    columns=poly.get_feature_names_out(X_train_transformed.columns)
)

print(f"Original features: {X_train_transformed.shape[1]}")
print(f"Polynomial features (degree=2): {poly_features_df.shape[1]}")
print(f"Feature expansion factor: {poly_features_df.shape[1] / X_train_transformed.shape[1]:.1f}x")
print()
print(f"First 5 new column names (of {poly_features_df.shape[1]}):")
for col in poly_features_df.columns[:5]:
    print(f"  {col}")
print(f"  ... and {poly_features_df.shape[1] - 5} more")
print()
print("First 4 rows (subset of columns):")
print(poly_features_df.iloc[:4, :6].to_string())

Original features: 9
Polynomial features (degree=2): 55
Feature expansion factor: 6.1x

First 5 new column names (of 55):
  1
  Age_scaled
  Salary_scaled
  Experience_scaled
  Department_Finance
  ... and 50 more

First 4 rows (subset of columns):
     1  Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR
0  1.0    0.834298      -0.867451          -1.559779            -0.57735      -0.480384
1  1.0   -0.689202       1.282818          -0.150946            -0.57735       2.081666
2  1.0    0.544107      -1.173357          -1.761041            -0.57735      -0.480384
3  1.0   -1.342131       0.821525          -0.150946            -0.57735      -0.480384


**KBinsDiscretizer():** Converts continuous variables into discrete, categorical buckets based on uniform, quantile, or kmeans strategies.

In [16]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")

binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)

print(f"Bin edges per feature (first 3 features):")
for i, col in enumerate(X_train_transformed.columns[:3]):
    print(f"  {col}: {np.round(kbins.bin_edges_[i], 3)}")
print()
print("Binned values (0, 1, or 2 for 3 bins):")
print(binned_df.head(8).to_string())
print()
print("Value counts for first feature:")
print(binned_df.iloc[:, 0].value_counts().sort_index().to_string())

Bin edges per feature (first 3 features):
  Age_scaled: [-1.415 -0.472  0.472  1.415]
  Salary_scaled: [-1.48  -0.432  0.617  1.665]
  Experience_scaled: [-1.761 -0.688  0.386  1.459]

Binned values (0, 1, or 2 for 3 bins):
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager
0         2.0            0.0                0.0                 0.0            0.0            2.0               0.0              0.0               0.0
1         0.0            2.0                1.0                 0.0            2.0            0.0               0.0              0.0               2.0
2         2.0            0.0                0.0                 0.0            0.0            2.0               0.0              0.0               2.0
3         0.0            2.0                1.0                 0.0            0.0            0.0               2.0              0.0               0.0
4         2.0        

### **Selecting Relevant Features**

Ideally, the optimal ML model is that one that performs the best with the fewest number of necessary
features. Simplicity is always favored over complexity, so we should explore techniques for identifying
features in our dataset that might not be helpful to model training so they can be removed.

**RFE (Recursive Feature Elimination):** Uses a model's importance ranking to recursively prune the least important features until a desired number remains.

In [17]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=1  # Rank all features
)

rfe.fit(X_train_transformed, y_train)

# Display feature rankings (1 = best)
ranking_df = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("RFE Feature Rankings (1 = most important):")
print(ranking_df.to_string(index=False))
print(f"\nBest feature: {ranking_df.iloc[0]['Feature']}")

RFE Feature Rankings (1 = most important):
           Feature  Ranking  Selected
  Position_Manager        1      True
   Position_Junior        2     False
     Salary_scaled        3     False
     Department_IT        4     False
        Age_scaled        5     False
     Department_HR        6     False
Department_Finance        7     False
 Experience_scaled        8     False
  Department_Sales        9     False

Best feature: Position_Manager


**SelectFromModel():** Selects features based on importance weights derived directly from an estimator (especially useful for tree-based models).

In [18]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold='mean'  # Keep features with importance > mean importance
)

selector.fit(X_train_transformed, y_train)

# Get results
selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

feature_importance = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance (|coef|)': np.abs(selector.estimator_.coef_),
    'Selected': selected_features_mask
}).sort_values('Importance (|coef|)', ascending=False)

print("Feature importance and selection:")
print(feature_importance.to_string(index=False))
print(f"\nThreshold (mean importance): {np.mean(np.abs(selector.estimator_.coef_)):.6f}")
print(f"Features selected: {len(selected_features)} of {X_train_transformed.shape[1]}")
print(f"Selected: {selected_features}")

Feature importance and selection:
           Feature  Importance (|coef|)  Selected
  Position_Manager         5.000000e-01      True
   Position_Junior         4.330127e-01      True
 Experience_scaled         1.191755e-15     False
     Salary_scaled         9.298118e-16     False
        Age_scaled         6.755007e-16     False
Department_Finance         5.342948e-16     False
     Department_HR         2.983724e-16     False
     Department_IT         2.810252e-16     False
  Department_Sales         1.561251e-16     False

Threshold (mean importance): 0.103668
Features selected: 2 of 9
Selected: ['Position_Junior', 'Position_Manager']


## **7. Practical Exercises on Data Preprocessing**
To synthesize these concepts, the document recommends building a comprehensive pipeline using the scikit-learn built-in California Housing dataset. The general workflow requires you to:

1. Load the data.
2. Split the data.
3. Construct a pipeline with at least three steps (e.g., imputation, scaling, and a final estimator for predicting the target house values) .
4. Fit the pipeline to the training data and evaluate it on the testing data.

A quick decision tree from the text notes: If data has outliers, consider RobustScaler. If it's normally distributed, use StandardScaler or MinMaxScaler. For skewed data, look at Power Transformers, and for sparse data, rely on MaxAbsScaler or Normalizer .